### AI 기반 안산시 독거노인 복지 사각지대 분석 및 복지 우선지원지역 추천

In [ ]:
# 원본 로컬 경로는 제거했습니다. 모듈화된 실행 코드는 src/welfare_analysis/를 참고하세요.


In [ ]:
print(len(elder['행정동'].unique()))
print(elder['행정동'].unique())

## 1. 동 이름 추출 함수

In [ ]:
def extract_dong(text):
    text = str(text)
    
    # 괄호 안 동 추출 예: (사동), (선부동)
    m = re.search(r"\(([^()]*동)\)", text)
    if m:
        return m.group(1).strip()
    
    # 주소 안의 ~동 추출
    m = re.search(r"([가-힣0-9]+동)", text)
    if m:
        return m.group(1).strip()
    
    return np.nan

## 2. 노인·독거노인 데이터 정리

In [ ]:
elder["노인수"] = elder["노인(남)"] + elder["노인(여)"]
elder["독거노인수"] = elder["독거노인(남)"] + elder["독거노인(여)"]

elder["독거노인비율"] = (
    elder["독거노인수"] / elder["노인수"] * 100
)

elder_base = elder[
    ["행정동", "노인수", "독거노인수", "독거노인비율"]
].copy()

elder_base.head()

## 3. 시설 데이터 행정동별 집계

In [ ]:
base_line = elder['행정동'].tolist()

In [ ]:
base_line

In [ ]:
print(len(base_line))

In [ ]:
medical.columns

In [ ]:
medical

In [ ]:
medical["행정동"] = medical["주소"].apply(extract_dong)

medical_count = (
    medical
    .dropna(subset=["행정동"])
    .groupby("행정동")
    .size()
    .reset_index(name="의료기관수")
)

medical_count.head()

In [ ]:
medical

In [ ]:
medical_list = medical_count['행정동'].tolist()

In [ ]:
medical_list

In [ ]:
print(len(medical_list))

In [ ]:
set(medical_list) - set(base_line)

In [ ]:
senior_facility["행정동"] = senior_facility["소재지도로명주소"].apply(extract_dong)

senior_count = (
    senior_facility
    .dropna(subset=["행정동"])
    .groupby("행정동")
    .size()
    .reset_index(name="노인복지시설수")
)

senior_count.head()

In [ ]:
senior_facility

In [ ]:
senior_facility["행정동"] = senior_facility["소재지도로명주소"].apply(extract_dong)

senior_count = (
    senior_facility
    .dropna(subset=["행정동"])
    .groupby("행정동")
    .size()
    .reset_index(name="노인복지시설수")
)

senior_count.head()

In [ ]:
senior_list = senior_facility['행정동'].tolist()

In [ ]:
set(senior_list) - set(base_line)

In [ ]:
welfare["행정동"] = welfare["소재지도로명주소"].apply(extract_dong)

welfare_count = (
    welfare
    .dropna(subset=["행정동"])
    .groupby("행정동")
    .size()
    .reset_index(name="사회복지관수")
)

welfare_count.head()

In [ ]:
welfare_list = welfare_count['행정동'].tolist()

In [ ]:
set(welfare_list) - set(base_line)

In [ ]:
health["행정동"] = health["소재지도로명주소"].apply(extract_dong)

health_count = (
    health
    .dropna(subset=["행정동"])
    .groupby("행정동")
    .size()
    .reset_index(name="보건시설수")
)

health_count.head()

In [ ]:
health

In [ ]:
health_list = health['행정동'].tolist()

In [ ]:
set(health_list) - set(base_line)

In [ ]:
park_count = (
    park
    .dropna(subset=["행정읍면동"])
    .groupby("행정읍면동")
    .size()
    .reset_index(name="공원수")
    .rename(columns={"행정읍면동": "행정동"})
)

park_count.head()

In [ ]:
park_count

In [ ]:
park_list = park_count['행정동'].tolist()

In [ ]:
set(park_list) - set(base_line)

In [ ]:
medicine["행정동"] = medicine["소재지도로명주소"].apply(extract_dong)

medicine_count = (
    medicine
    .dropna(subset=["행정동"])
    .groupby("행정동")
    .size()
    .reset_index(name="상비약판매업소수")
)

medicine_count.head()

In [ ]:
medicine_list = medicine_count['행정동'].tolist()

In [ ]:
set(medicine_list) - set(base_line)

In [ ]:
dong_df = elder_base.copy()

for df_count in [
    medical_count,
    senior_count,
    welfare_count,
    health_count,
    park_count,
    medicine_count
]:
    dong_df = dong_df.merge(
        df_count,
        on="행정동",
        how="left"
    )

# 시설 없는 곳은 0 처리
facility_cols = [
    "의료기관수",
    "노인복지시설수",
    "사회복지관수",
    "보건시설수",
    "공원수",
    "상비약판매업소수"
]

dong_df[facility_cols] = dong_df[facility_cols].fillna(0)

dong_df.head()

In [ ]:
dong_df["노인천명당_의료기관수"] = dong_df["의료기관수"] / dong_df["노인수"] * 1000
dong_df["노인천명당_노인복지시설수"] = dong_df["노인복지시설수"] / dong_df["노인수"] * 1000
dong_df["노인천명당_사회복지관수"] = dong_df["사회복지관수"] / dong_df["노인수"] * 1000
dong_df["노인천명당_보건시설수"] = dong_df["보건시설수"] / dong_df["노인수"] * 1000
dong_df["노인천명당_공원수"] = dong_df["공원수"] / dong_df["노인수"] * 1000
dong_df["노인천명당_상비약판매업소수"] = dong_df["상비약판매업소수"] / dong_df["노인수"] * 1000

dong_df.head()

In [ ]:
features = [
    "노인수",
    "독거노인수",
    "독거노인비율",
    "노인천명당_의료기관수",
    "노인천명당_노인복지시설수",
    "노인천명당_사회복지관수",
    "노인천명당_보건시설수",
    "노인천명당_공원수",
    "노인천명당_상비약판매업소수"
]

X = dong_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

dong_df["cluster"] = kmeans.fit_predict(X_scaled)

dong_df.head()

In [ ]:
cluster_summary = (
    dong_df
    .groupby("cluster")[features]
    .mean()
    .round(2)
)

cluster_summary

In [ ]:
# 독거노인 수는 많고, 복지·보건·의료 시설은 적은 지역 찾기
dong_df["복지시설합계"] = (
    dong_df["의료기관수"]
    + dong_df["노인복지시설수"]
    + dong_df["사회복지관수"]
    + dong_df["보건시설수"]
)

dong_df["노인천명당_복지시설합계"] = (
    dong_df["복지시설합계"] / dong_df["노인수"] * 1000
)

priority = dong_df.sort_values(
    ["독거노인수", "노인천명당_복지시설합계"],
    ascending=[False, True]
)

priority[
    [
        "행정동",
        "노인수",
        "독거노인수",
        "독거노인비율",
        "복지시설합계",
        "노인천명당_복지시설합계",
        "cluster"
    ]
].head(10)

## 내가 하는거

In [ ]:
elder

In [ ]:
print(len(elder['행정동'].unique()))

In [ ]:
elder['행정동']

In [ ]:
import geopandas as gpd
gdf = gpd.read_file("LSMD_ADM_SECT_UMD_41_202606.shp",encoding="cp949")
gdf

In [ ]:
ansan = gdf[
    gdf["COL_ADM_SE"].astype(str).isin(["41271", "41273"])
].copy()

ansan[["EMD_CD", "COL_ADM_SE", "EMD_NM"]]

In [ ]:
import geopandas as gpd
gdf = gpd.read_file("BND_ADM_DONG_PG.shp",encoding="cp949")
gdf

In [ ]:
# 안산시 행정동 25개
ansan_dongs = [
    "일동", "이동", "사동", "사이동", "해양동",
    "본오1동", "본오2동", "본오3동",
    "부곡동", "월피동", "성포동", "반월동", "안산동",
    "와동", "고잔동", "중앙동", "호수동", "원곡동", "백운동",
    "신길동", "초지동",
    "선부1동", "선부2동", "선부3동",
    "대부동"
]

# 안산시 25개 행정동만 추출
ansan_gdf = gdf[gdf["ADM_NM"].isin(ansan_dongs)].copy()

# 추출 결과 확인
print("추출된 행정동 개수:", ansan_gdf["ADM_NM"].nunique())
print(sorted(ansan_gdf["ADM_NM"].unique()))

# 누락된 행정동 확인
missing = set(ansan_dongs) - set(ansan_gdf["ADM_NM"].unique())
print("누락된 행정동:", missing)

In [ ]:
ansan_gdf

In [ ]:
print(len(ansan_gdf))

In [ ]:
gdf[gdf["ADM_NM"]=="사동"][["ADM_CD","ADM_NM"]]

In [ ]:
# 안산시 행정동 25개
ansan_dongs = [
    "일동", "이동", "사동", "사이동", "해양동",
    "본오1동", "본오2동", "본오3동",
    "부곡동", "월피동", "성포동", "반월동", "안산동",
    "와동", "고잔동", "중앙동", "호수동", "원곡동", "백운동",
    "신길동", "초지동",
    "선부1동", "선부2동", "선부3동",
    "대부동"
]

# ADM_CD를 문자열로 변환
gdf["ADM_CD"] = gdf["ADM_CD"].astype(str)

# 안산시 코드 3109 + 행정동명 조건
ansan_gdf = gdf[
    (gdf["ADM_CD"].str.startswith("3109")) &
    (gdf["ADM_NM"].isin(ansan_dongs))
].copy()

print("추출된 행정동 수:", len(ansan_gdf))
print(sorted(ansan_gdf["ADM_NM"].unique()))

missing = set(ansan_dongs) - set(ansan_gdf["ADM_NM"].unique())
print("누락된 행정동:", missing)

In [ ]:
ansan_gdf

In [ ]:
print(len(ansan_gdf))

In [ ]:
# 위경도 좌표계로 변환
ansan_gdf = ansan_gdf.to_crs(epsg=4326)

# 대표점 생성
ansan_gdf["대표점"] = ansan_gdf.geometry.representative_point()

# 위도/경도 생성
ansan_gdf["위도"] = ansan_gdf["대표점"].y
ansan_gdf["경도"] = ansan_gdf["대표점"].x

coord_df = ansan_gdf[
    ["ADM_CD", "ADM_NM", "위도", "경도"]
].copy()

coord_df = coord_df.rename(columns={
    "ADM_NM": "행정동"
})

coord_df

In [ ]:
dong_df = dong_df.merge(
    coord_df[["행정동","위도","경도"]],
    on="행정동",
    how="left"
)

In [ ]:
dong_df

In [ ]:
import folium
import numpy as np

# 1. dong_df에 행정동 위도/경도 붙이기
dong_map = dong_df.merge(
    coord_df[["행정동", "위도", "경도"]],
    on="행정동",
    how="left"
)

dong_map

In [ ]:
# coord_df에서 새로 붙은 좌표를 최종 좌표로 사용
dong_map["위도"] = dong_map["위도_y"]
dong_map["경도"] = dong_map["경도_y"]

# 불필요한 컬럼 제거
dong_map = dong_map.drop(
    columns=["위도_x", "경도_x", "위도_y", "경도_y"]
)

# 확인
dong_map[["행정동", "위도", "경도"]]

In [ ]:
import folium

m = folium.Map(
    location=[37.3219, 126.8309],
    zoom_start=11
)

for _, row in dong_map.iterrows():
    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=max(6, row["독거노인수"] / 180),
        popup=(
            f"<b>{row['행정동']}</b><br>"
            f"노인수: {int(row['노인수'])}명<br>"
            f"독거노인수: {int(row['독거노인수'])}명<br>"
            f"독거노인비율: {row['독거노인비율']:.1f}%<br>"
            f"복지시설합계: {int(row['복지시설합계'])}개<br>"
            f"노인천명당 복지시설: {row['노인천명당_복지시설합계']:.2f}개"
        ),
        fill=True,
        fill_opacity=0.7
    ).add_to(m)

m

In [ ]:
# map_gdf 다시 정의
map_gdf = ansan_gdf.merge(
    dong_map,
    left_on="ADM_NM",
    right_on="행정동",
    how="left"
).to_crs(epsg=4326)

# Point 컬럼 제거
for col in ["대표점", "center", "centroid"]:
    if col in map_gdf.columns:
        map_gdf = map_gdf.drop(columns=[col])

# geometry 외에 shapely 객체 컬럼이 또 있는지 제거
for col in map_gdf.columns:
    if col != "geometry":
        if map_gdf[col].apply(lambda x: hasattr(x, "geom_type")).any():
            map_gdf = map_gdf.drop(columns=[col])

# cluster를 int로 정리
map_gdf["cluster"] = map_gdf["cluster"].astype(int)

In [ ]:
# 위도_y, 경도_y를 최종 위도/경도로 정리
map_gdf["위도"] = map_gdf["위도_y"]
map_gdf["경도"] = map_gdf["경도_y"]

# 불필요한 중복 좌표 컬럼 제거
map_gdf = map_gdf.drop(
    columns=["위도_x", "경도_x", "위도_y", "경도_y"],
    errors="ignore"
)

print(map_gdf[["행정동", "위도", "경도"]])

In [ ]:
for _, row in map_gdf.iterrows():
    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=max(5, row["독거노인수"] / 250),
        popup=(
            f"<b>{row['행정동']}</b><br>"
            f"노인수: {int(row['노인수'])}명<br>"
            f"독거노인수: {int(row['독거노인수'])}명<br>"
            f"독거노인비율: {row['독거노인비율']:.1f}%<br>"
            f"복지시설합계: {int(row['복지시설합계'])}개<br>"
            f"노인천명당 복지시설: {row['노인천명당_복지시설합계']:.2f}개<br>"
            f"AI 군집: {row['cluster']}"
        ),
        color="blue",
        fill=True,
        fill_opacity=0.8
    ).add_to(m)

m

In [ ]:
import folium

cluster_colors = {
    0: "#2ECC71",   # 초록
    1: "#F1C40F",   # 노랑
    2: "#3498DB",   # 파랑
    3: "#E74C3C"    # 빨강
}

m2 = folium.Map(
    location=[37.3219,126.8309],
    zoom_start=11,
    tiles="cartodbpositron"
)

folium.GeoJson(

    map_gdf,

    style_function=lambda feature:{

        "fillColor":cluster_colors[
            feature["properties"]["cluster"]
        ],

        "color":"black",
        "weight":1,

        "fillOpacity":0.7

    },

    tooltip=folium.GeoJsonTooltip(

        fields=[
            "행정동",
            "노인수",
            "독거노인수",
            "복지시설합계",
            "cluster"
        ],

        aliases=[
            "행정동",
            "노인수",
            "독거노인수",
            "복지시설",
            "AI 군집"
        ]

    )

).add_to(m2)

m2

In [ ]:
print(dong_map.columns.tolist())
print(map_gdf.columns.tolist())

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

pca_col = "PCA_복지취약도"

pca_features = [
    "독거노인수",
    "독거노인비율",
    "노인천명당_복지시설합계",
    "노인천명당_의료기관수"
]

X = dong_map[pca_features].replace([np.inf, -np.inf], np.nan).fillna(0).copy()

# 시설이 많을수록 취약도는 낮아져야 하므로 방향 반전
X["노인천명당_복지시설합계"] *= -1
X["노인천명당_의료기관수"] *= -1

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=1)
dong_map[pca_col] = pca.fit_transform(X_scaled)

dong_map[["행정동", pca_col]].sort_values(pca_col, ascending=False)

In [ ]:
map_gdf = ansan_gdf.merge(
    dong_map,
    left_on="ADM_NM",
    right_on="행정동",
    how="left"
).to_crs(epsg=4326)

for col in ["대표점", "center", "centroid"]:
    if col in map_gdf.columns:
        map_gdf = map_gdf.drop(columns=[col])

# 좌표 컬럼 정리
if "위도_y" in map_gdf.columns:
    map_gdf["위도"] = map_gdf["위도_y"]
    map_gdf["경도"] = map_gdf["경도_y"]

In [ ]:
map_gdf

In [ ]:
import folium
import branca.colormap as cm

# PCA 컬럼명 확인 후 맞춰주세요
# 예: "PCA_복지취약도" 또는 "PCA_복지취약도지수"
pca_col = "PCA_복지취약도"

# 행정동 경계 + PCA 결과 결합
map_gdf_pca = ansan_gdf.merge(
    dong_map,
    left_on="ADM_NM",
    right_on="행정동",
    how="left"
).to_crs(epsg=4326)

# Point 컬럼 제거
for col in ["대표점", "center", "centroid"]:
    if col in map_gdf_pca.columns:
        map_gdf_pca = map_gdf_pca.drop(columns=[col])

# 색상 범위
min_val = map_gdf_pca[pca_col].min()
max_val = map_gdf_pca[pca_col].max()

colormap = cm.linear.YlOrRd_09.scale(min_val, max_val)
colormap.caption = "PCA 기반 복지취약도"

m_pca = folium.Map(
    location=[37.3219, 126.8309],
    zoom_start=11,
    tiles="cartodbpositron"
)

folium.GeoJson(
    map_gdf_pca,
    name="PCA 기반 복지취약도",
    style_function=lambda feature: {
        "fillColor": colormap(feature["properties"][pca_col])
        if feature["properties"][pca_col] is not None
        else "gray",
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "행정동",
            "노인수",
            "독거노인수",
            "독거노인비율",
            "노인천명당_복지시설합계",
            "노인천명당_의료기관수",
            pca_col,
            "cluster"
        ],
        aliases=[
            "행정동",
            "노인수",
            "독거노인수",
            "독거노인비율(%)",
            "노인천명당 복지시설",
            "노인천명당 의료기관",
            "PCA 복지취약도",
            "AI 군집"
        ],
        localize=True
    )
).add_to(m_pca)

colormap.add_to(m_pca)
folium.LayerControl().add_to(m_pca)

m_pca

In [ ]:
pca_top10 = (
    dong_map
    .sort_values(pca_col, ascending=False)
    .head(10)
)

pca_top10[
    [
        "행정동",
        "노인수",
        "독거노인수",
        "독거노인비율",
        "노인천명당_복지시설합계",
        "노인천명당_의료기관수",
        pca_col,
        "cluster"
    ]
]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.family'] = 'Malgun Gothic'   # 맑은 고딕
matplotlib.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(10, 6))

plt.barh(
    pca_top10["행정동"],
    pca_top10[pca_col]
)

plt.gca().invert_yaxis()
plt.xlabel("PCA 기반 복지취약도")
plt.ylabel("행정동")
plt.title("PCA 기반 복지취약도 TOP10")
plt.show()

In [ ]:
print(pca.explained_variance_ratio_)

In [ ]:
loading = pd.DataFrame(
    pca.components_.T,
    index=pca_features,
    columns=["PC1"]
)

loading

## AI 추천 가상 시뮬레이션

In [ ]:
def recalc_facility(df):
    df = df.copy()

    df["복지시설합계"] = (
        df["의료기관수"]
        + df["노인복지시설수"]
        + df["사회복지관수"]
        + df["보건시설수"]
    )

    df["노인천명당_복지시설합계"] = df["복지시설합계"] / df["노인수"] * 1000
    df["노인천명당_의료기관수"] = df["의료기관수"] / df["노인수"] * 1000

    return df


def simulate_rank_change(target_dong, facility_col="노인복지시설수", add_count=3):
    sim = dong_map.copy()

    # 시설 추가
    sim.loc[sim["행정동"] == target_dong, facility_col] += add_count

    # 시설 지표 재계산
    sim = recalc_facility(sim)

    # 기존 PCA 기준으로 변환
    X_sim = make_pca_input(sim)
    X_sim_scaled = scaler.transform(X_sim)
    sim["시뮬레이션_PCA"] = pca.transform(X_sim_scaled)

    sim["시뮬레이션순위"] = sim["시뮬레이션_PCA"].rank(
        ascending=False,
        method="min"
    ).astype(int)

    before_rank = dong_map.loc[dong_map["행정동"] == target_dong, "현재순위"].values[0]
    after_rank = sim.loc[sim["행정동"] == target_dong, "시뮬레이션순위"].values[0]

    before_pca = dong_map.loc[dong_map["행정동"] == target_dong, pca_col].values[0]
    after_pca = sim.loc[sim["행정동"] == target_dong, "시뮬레이션_PCA"].values[0]

    return {
        "행정동": target_dong,
        "추가시설": facility_col,
        "추가개수": add_count,
        "시행전_PCA": before_pca,
        "시행후_PCA": after_pca,
        "현재순위": before_rank,
        "시행후순위": after_rank,
        "순위개선": before_rank - after_rank
    }

In [ ]:
def make_pca_input(df):
    X = df[pca_features].replace([np.inf, -np.inf], np.nan).fillna(0).copy()

    # 시설이 많을수록 취약도는 낮아져야 하므로 방향 반전
    X["노인천명당_복지시설합계"] *= -1
    X["노인천명당_의료기관수"] *= -1

    return X

In [ ]:
pca_col = "PCA_복지취약도"

# 만약 기존 컬럼명이 PCA_라면 아래 줄 사용
# pca_col = "PCA_"

dong_map["현재순위"] = dong_map[pca_col].rank(
    ascending=False,
    method="min"
).astype(int)

dong_map[["행정동", pca_col, "현재순위"]].sort_values("현재순위").head(10)

In [ ]:
results = []

for dong in dong_map["행정동"]:
    results.append(
        simulate_rank_change(
            target_dong=dong,
            facility_col="노인복지시설수",
            add_count=3
        )
    )


rank_simulation = pd.DataFrame(results)

rank_simulation = rank_simulation.sort_values(
    ["순위개선", "현재순위"],
    ascending=[False, True]
)

rank_simulation

In [ ]:
rank_simulation.head(10)

## 복지시설 추가 개수 별 시나리오

In [ ]:
# 복지시설 1개, 2개, 3개 추가 시나리오 비교

scenario_results = []

for add_count in [1, 2, 3]:
    for dong in dong_map["행정동"]:
        scenario_results.append(
            simulate_rank_change(
                target_dong=dong,
                facility_col="노인복지시설수",
                add_count=add_count
            )
        )

scenario_df = pd.DataFrame(scenario_results)

# 개선폭 계산
# 현재순위 20위 -> 시행후순위 10위면 10단계 개선
scenario_df["개선폭"] = scenario_df["현재순위"] - scenario_df["시행후순위"]

scenario_df = scenario_df.sort_values(
    ["추가개수", "개선폭"],
    ascending=[True, False]
)

scenario_df

In [ ]:
scenario_summary = (
    scenario_df
    .groupby("추가개수")
    .agg(
        평균개선폭=("개선폭", "mean"),
        최대개선폭=("개선폭", "max"),
        개선지역수=("개선폭", lambda x: (x > 0).sum())
    )
    .reset_index()
)

scenario_summary

In [ ]:
top_by_scenario = (
    scenario_df
    .sort_values(["추가개수", "개선폭", "현재순위"], ascending=[True, False, True])
    .groupby("추가개수")
    .head(5)
)

top_by_scenario[
    [
        "행정동",
        "추가시설",
        "추가개수",
        "시행전_PCA",
        "시행후_PCA",
        "현재순위",
        "시행후순위",
        "개선폭"
    ]
]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

plt.figure(figsize=(8, 5))

plt.bar(
    scenario_summary["추가개수"].astype(str) + "개 추가",
    scenario_summary["평균개선폭"]
)

plt.xlabel("노인복지시설 추가 개수")
plt.ylabel("평균 순위 개선폭")
plt.title("노인복지시설 추가 개수별 평균 복지취약도 순위 개선 효과")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()